In [ ]:
"""
Fine-tune the model's trainable `alpha` (text-attention blending gate).

The goal: only update the gate parameters that control how much the model trusts
external text attention when predicting *test rows from train context*.

This script is written in a step-by-step style. Each step is explained in code
and comments so you can port it to a notebook cell-by-cell.

What the gate is in THIS repo
-----------------------------
In `tabdpt/model.py`, the transformer stack is set up as:
- layers 0..(L-2): normal attention
- last layer: `text_enhanced=True` and has `alpha` (per-head logits)

During the last layer's datapoint attention:
- train rows attend to train rows (no external attention)
- test rows attend to train rows, optionally mixing in text similarity attention
  computed from text embeddings

The gate parameter stored on the last block is a vector of logits with shape (H,),
one per attention head. The forward pass applies `sigmoid()` to obtain values in
(0, 1), which are then used as a blending coefficient.
"""

from __future__ import annotations

import ast

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    import schedulefree  # type: ignore
except Exception:  # noqa: BLE001
    schedulefree = None

from tabdpt import TabDPTRegressor
from tabdpt.utils import generate_random_permutation, pad_x


# -----------------------------
# Step 0) Reproducibility
# -----------------------------
torch.manual_seed(0)
np.random.seed(0)


# -----------------------------
# Step 1) Configuration
# -----------------------------
# DATA_PATH = "data/climate_ttc/climate_2014_2023_final_with_embeddings_lag_3.csv"
# DATE_COLUMN = "date"
# TARGET_COLUMN = "temp"
# NUMERIC_FEATURES = ["precip_lag0","precip_lag1","precip_lag2","precip_lag3",
#                     "humidity_lag0","humidity_lag1","humidity_lag2","humidity_lag3",
#                     "windspeed_lag0","windspeed_lag1","windspeed_lag2","windspeed_lag3",
#                     "temp_lag0","temp_lag1","temp_lag2","temp_lag3"]

DATA_PATH = "data/bitcoin/bitcoin_final_with_embeddings_lag_3.csv"
DATE_COLUMN = "Date"
NUMERIC_FEATURES = ["Open_lag1", "Open_lag2", "Open_lag3",
                    "High_lag1", "High_lag2", "High_lag3",
                    "Low_lag1", "Low_lag2", "Low_lag3",
                    "Close_lag1", "Close_lag2", "Close_lag3",
                    "Adj_Close_lag1", "Adj_Close_lag2", "Adj_Close_lag3"]
TARGET_COLUMN="Adj_Close"


# Text lags to use (your regressor averages similarity across the L dimension).
TEXT_EMBEDDING_LAGS = [1, 2, 3]
# Either provide an explicit list of embedding columns, or a template for lags.
# If EMBEDDING_COLUMNS is not None, it takes precedence.
EMBEDDING_COLUMNS = None
EMBEDDING_COLUMN_TEMPLATE = "embedding_summary_gpt-5-mini_lag{lag}"

# Use chronological splits to avoid leakage in time series.
MAX_ROWS = None
# Fraction-based split: first CONTEXT_RATIO for context, next TUNE_RATIO for tuning, remainder eval.
CONTEXT_RATIO = 0.2
TUNE_RATIO = 0.6
# Base context size used inside the rolling/batching fine-tune loop.
# Following the example: if N_tune >= 200, we use base 200; else we use half of tune.
BASE_CONTEXT_IN_TUNE = 200
# How many new tune rows to add per batch step.
TUNE_BATCH_SIZE = 4
# Optional: cap the training context length during tuning to avoid OOM.
# If None, use all available context; if an int, keep only the last K rows of
# [global_context + past_tune] when building each training window.
MAX_CONTEXT_FOR_TUNE = 1000

# Model loading / inference options.
# Set MODEL_WEIGHT_PATH to a local .safetensors file to avoid HF downloads.
MODEL_WEIGHT_PATH = None
DEVICE = None  # e.g., "cuda:0" or "cpu"
USE_FLASH = True
COMPILE_MODEL = True
N_ENSEMBLES = 1
FEATURE_SUBSAMPLE_SEED = 0

# Fine-tuning hyperparameters (we tune only a few numbers: per-head gate logits).
EPOCHS = 50
LEARNING_RATE = 1e-4  # kept for backward compatibility; see GATE_LR below
LOG_EVERY = 5  # epoch-level logging cadence
STEP_LOG_EVERY = 10  # step-level logging inside each epoch

# Diagnostics: compare predictions with/without text attention on a slice.
DEBUG_TEXT_EFFECT = True
DEBUG_TEXT_EFFECT_ROWS = 200

# Use a higher LR for the gate to encourage it to move away from ~0.5 if helpful.
GATE_LR = 5e-1

# Optional: clamp gate *logits* to avoid extreme saturation of sigmoid.
GATE_LOGIT_CLAMP = 10.0

# Optional: encourage gate probabilities away from 0.5 (toward extremes).
# Minimizing gate*(1-gate) pushes sigmoids toward 0 or 1. Set to 0.0 to disable.
GATE_REG_STRENGTH = 0

# Reuse the no-text baseline metrics after tuning to avoid nondeterministic drift.
FREEZE_NO_TEXT_BASELINE = True


# -----------------------------
# Step 2) Data loading utilities
# -----------------------------

def load_tabdpt_regressor(
    *,
    device: str | None,
    model_weight_path: str | None,
    text_enhanced: bool = True,
) -> TabDPTRegressor:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    return TabDPTRegressor(
        device=device,
        text_enhanced=text_enhanced,
        model_weight_path=model_weight_path,
        use_flash=USE_FLASH,
        compile=COMPILE_MODEL,
    )


def prepare_feature_reduction(
    reg: TabDPTRegressor,
    *,
    seed: int | None,
) -> tuple[str | None, np.ndarray | None]:
    if reg.n_features <= reg.max_features:
        return None, None
    if reg.feature_reduction == "pca":
        if not hasattr(reg, "V"):
            raise RuntimeError("PCA reduction requested but reg.V is missing.")
        return "pca", reg.V.detach().cpu().numpy()
    if reg.feature_reduction == "subsample":
        perm = generate_random_permutation(reg.n_features, seed=seed).cpu().numpy()
        return "subsample", perm
    raise ValueError(f"Unsupported feature_reduction: {reg.feature_reduction}")


def preprocess_features(
    reg: TabDPTRegressor,
    X: np.ndarray,
    *,
    reduction_mode: str | None,
    reduction_payload: np.ndarray | None,
) -> np.ndarray:
    X_proc = X
    if reg.missing_indicators:
        inds = np.isnan(X_proc)[:, reg.has_missing_indicator].astype(float)
        X_proc = np.hstack((X_proc, inds))
    X_proc = reg.imputer.transform(X_proc)
    if reg.scaler:
        X_proc = reg.scaler.transform(X_proc)
        if reg.normalizer == "quantile-uniform":
            X_proc = 2 * X_proc - 1

    if reduction_mode == "pca":
        if reduction_payload is None:
            raise ValueError("PCA reduction requested without a payload.")
        X_proc = X_proc @ reduction_payload
    elif reduction_mode == "subsample":
        if reduction_payload is None:
            raise ValueError("Subsample reduction requested without a payload.")
        X_proc = X_proc[:, reduction_payload][:, :reg.max_features]

    return X_proc.astype(np.float32)


def _parse_embedding_column(series: pd.Series) -> np.ndarray:
    """Parse a CSV embedding column (stringified list) to a float32 array (N, D)."""
    parsed = [np.asarray(ast.literal_eval(v), dtype=np.float32) for v in series.astype(str).tolist()]
    return np.stack(parsed, axis=0)


def load_climate_dataset(
    *,
    path: str,
    max_rows: int | None,
    embedding_lags: list[int],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load numeric features + target + text embeddings.

    Returns:
    - X: (N, F) numeric features
    - y: (N,) target
    - text: (N, L, D) text embeddings (lags are treated as separate text features)
    """
    df = pd.read_csv(path)

    # Sort by date to make chronological splits meaningful.
    if DATE_COLUMN in df.columns:
        df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
        df = df.sort_values(DATE_COLUMN).reset_index(drop=True)

    if max_rows is not None:
        df = df.head(max_rows).reset_index(drop=True)

    for col in [*NUMERIC_FEATURES, TARGET_COLUMN]:
        if col not in df.columns:
            raise ValueError(f"Missing required column '{col}' in {path}")

    X = df[NUMERIC_FEATURES].astype(np.float32).to_numpy()
    y = df[TARGET_COLUMN].astype(np.float32).to_numpy()

    if EMBEDDING_COLUMNS is not None:
        embedding_cols = EMBEDDING_COLUMNS
    else:
        embedding_cols = [EMBEDDING_COLUMN_TEMPLATE.format(lag=lag) for lag in embedding_lags]
    for col in embedding_cols:
        if col not in df.columns:
            raise ValueError(f"Missing embedding column '{col}' in {path}")

    # Each lag becomes a separate text feature: (N, L, D)
    by_lag = [_parse_embedding_column(df[col]) for col in embedding_cols]  # list[(N, D)]
    text = np.stack(by_lag, axis=1)  # (N, L, D)
    return X, y, text


# -----------------------------
# Step 3) Chronological splitting
# -----------------------------
def time_split(
    X: np.ndarray,
    y: np.ndarray,
    text: np.ndarray,
    *,
    context_ratio: float,
    tune_ratio: float,
) -> tuple[np.ndarray, ...]:
    """
    Split sequentially into context / tune / eval.

    - context: first `context_ratio` fraction (chronological)
    - tune: next `tune_ratio` fraction
    - eval: remainder
    """
    n = len(y)
    n_context = int(n * context_ratio)
    n_tune = int(n * tune_ratio)
    n_eval = n - n_context - n_tune
    if n_context <= 0 or n_tune <= 0 or n_eval <= 0:
        raise ValueError(f"Invalid split sizes: n={n}, context={n_context}, tune={n_tune}, eval={n_eval}")

    X_context, y_context, text_context = X[:n_context], y[:n_context], text[:n_context]
    X_tune, y_tune, text_tune = (
        X[n_context : n_context + n_tune],
        y[n_context : n_context + n_tune],
        text[n_context : n_context + n_tune],
    )
    X_eval, y_eval, text_eval = X[n_context + n_tune :], y[n_context + n_tune :], text[n_context + n_tune :]

    return (
        X_context,
        y_context,
        text_context,
        X_tune,
        y_tune,
        text_tune,
        X_eval,
        y_eval,
        text_eval,
    )


# -----------------------------
# Step 4) Evaluation helper (uses reg.predict)
# -----------------------------
def _format_metrics(label: str, mae: float, rmse: float) -> None:
    print(f"{label} | MAE: {mae:.4f} | RMSE: {rmse:.4f}")


def _compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float]:
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    return mae, rmse


def evaluate(
    reg: TabDPTRegressor,
    *,
    X: np.ndarray,
    y: np.ndarray,
    text: np.ndarray,
    label: str,
) -> tuple[float, float]:
    """
    End-to-end evaluation using the regressor API.

    Important: `predict()` runs under `torch.no_grad()`, so it is for evaluation only.
    """
    preds = reg.predict(X, text=text, n_ensembles=N_ENSEMBLES)
    mae, rmse = _compute_metrics(y, preds)
    _format_metrics(label, mae, rmse)
    return mae, rmse


def evaluate_no_text_attn(
    reg: TabDPTRegressor,
    *,
    X: np.ndarray,
    y: np.ndarray,
    label: str,
) -> tuple[float, float]:
    """
    Baseline evaluation WITHOUT using any text attention.

    Implementation detail:
    - Passing `text=None` forces the model to ignore text-enhanced attention.
    """
    preds = reg.predict(X, text=None, n_ensembles=N_ENSEMBLES)
    mae, rmse = _compute_metrics(y, preds)
    _format_metrics(label, mae, rmse)
    return mae, rmse


def compare_text_effect(
    reg: TabDPTRegressor,
    *,
    X: np.ndarray,
    text: np.ndarray,
    label: str,
    max_rows: int | None = None,
) -> None:
    """
    Compare predictions with/without text attention to confirm text attention is active.
    """
    if max_rows is not None:
        X = X[:max_rows]
        text = text[:max_rows]

    preds_text = reg.predict(X, text=text, n_ensembles=N_ENSEMBLES)
    preds_no_text = reg.predict(X, text=None, n_ensembles=N_ENSEMBLES)
    diff = np.abs(preds_text - preds_no_text)
    print(f"{label} | Δpred(mean/max)={diff.mean():.6f}/{diff.max():.6f}")


# -----------------------------
# Step 5) Gate-only parameter selection
# -----------------------------
def get_last_layer_gate_param(reg: TabDPTRegressor) -> torch.nn.Parameter:
    """
    Return the trainable gate parameter (per-head logits) from the last transformer block.

    Only the last transformer block is `text_enhanced=True` in this repo.
    """
    last_block = reg.model.transformer_encoder[-1]
    if getattr(last_block, "alpha", None) is None:
        raise RuntimeError("Last block has no alpha gate. Is text_enhanced enabled in the model?")
    return last_block.alpha


def freeze_all_but_last_gate(reg: TabDPTRegressor) -> torch.nn.Parameter:
    """Freeze all parameters and enable gradients only for the last block's alpha gate."""
    for p in reg.model.parameters():
        p.requires_grad_(False)
    gate = get_last_layer_gate_param(reg)
    gate.requires_grad_(True)
    return gate


def gate_stats(gate_logits: torch.Tensor) -> str:
    """Human-readable summary for per-head gate logits and their sigmoid values."""
    logits = gate_logits.detach().float().cpu().reshape(-1)
    gate = torch.sigmoid(logits)
    sample_count = min(8, gate.numel())
    sample_vals = torch.round(gate[:sample_count], decimals=4).tolist()
    sample_str = f"{sample_vals}" if gate.numel() <= sample_count else f"{sample_vals}..."
    return (
        f"logits(mean/min/max)={logits.mean().item():.4f}/{logits.min().item():.4f}/{logits.max().item():.4f} | "
        f"sigmoid(mean/min/max)={gate.mean().item():.4f}/{gate.min().item():.4f}/{gate.max().item():.4f} | "
        f"sigmoid(sample)={sample_str}"
    )


# -----------------------------
# Step 6) Fine-tuning loop (calls reg.model directly)
# Rolling window: at step i, train on context + tune[0:i], predict tune[i].
# -----------------------------
def fine_tune_external_gate(
    reg: TabDPTRegressor,
    *,
    X_context_proc: np.ndarray,
    y_context: np.ndarray,
    text_context: np.ndarray,
    X_tune_proc: np.ndarray,
    y_tune: np.ndarray,
    text_tune: np.ndarray,
) -> None:
    """
    Fine-tune only the last layer's gate on the tune split.

    Why call the model directly?
    - `reg.predict()` disables gradients, so we can't optimize with it.
    - We still re-use everything from `reg.fit()`:
        - fitted imputers/scalers
        - text similarity attention via `_compute_attn_weight_pairwise_avg(...)`

    Rolling window (per step inside each epoch):
    - Train set = context + all prior tune rows
    - Test row = current tune row
    - Loss = MSE on that current row (raw space)
    """
    # (1) Select tunable params (gate only) and build optimizer.
    gate = freeze_all_but_last_gate(reg)
    # Use Schedule-Free AdamW when available; fall back to AdamW otherwise.
    if schedulefree is None:
        print("WARNING: `schedulefree` not installed; falling back to torch.optim.AdamW.")
        optimizer = torch.optim.AdamW([gate], lr=GATE_LR, weight_decay=0.0)
    else:
        # optimizer = schedulefree.AdamWScheduleFree(model.parameters(), lr=lr, weight_decay=0.0)
        optimizer = schedulefree.AdamWScheduleFree([gate], lr=GATE_LR, weight_decay=0.0)
    print("Tuning last-layer alpha gate:", gate_stats(gate))

    # (3) Rolling-window gradient loop (with base context then growing window)
    reg.model.eval()
    if hasattr(optimizer, "train"):
        optimizer.train()

    num_steps = int(np.ceil(len(y_tune) / TUNE_BATCH_SIZE))
    for epoch in range(1, EPOCHS + 1):
        for step_idx in range(num_steps):
            optimizer.zero_grad()

            start = step_idx * TUNE_BATCH_SIZE
            end = min(len(y_tune), start + TUNE_BATCH_SIZE)

            # Base context portion inside the tune set (e.g., first 200 tune rows) + accumulated history.
            base_limit = min(BASE_CONTEXT_IN_TUNE, len(y_tune))
            past_limit = max(base_limit, start)

            # Train set for this step: global context + tune[0:past_limit]
            X_train_full = np.concatenate((X_context_proc, X_tune_proc[:past_limit]))
            y_train_full = np.concatenate((y_context, y_tune[:past_limit]))
            text_train_full = np.concatenate((text_context, text_tune[:past_limit]), axis=0)

            # Apply optional context cap to avoid long sequences (OOM protection).
            if MAX_CONTEXT_FOR_TUNE is not None:
                X_train_step = X_train_full[-MAX_CONTEXT_FOR_TUNE:]
                y_train_step = y_train_full[-MAX_CONTEXT_FOR_TUNE:]
                text_train_step = text_train_full[-MAX_CONTEXT_FOR_TUNE:]
            else:
                X_train_step = X_train_full
                y_train_step = y_train_full
                text_train_step = text_train_full

            # Test batch for this step: tune[start:end]
            X_test_step = X_tune_proc[start:end]
            y_target = torch.tensor(y_tune[start:end], dtype=torch.float32, device=reg.device)

            train_text_batch = text_train_step[None, ...]
            test_text_batch = text_tune[start:end][None, ...]
            attn_weight_external = reg._compute_attn_weight_pairwise_avg(train_text_batch, test_text_batch)

            X_train_tensor = torch.tensor(X_train_step, dtype=torch.float32, device=reg.device).unsqueeze(0)
            X_train_tensor = pad_x(X_train_tensor, reg.max_features)
            X_test_tensor = torch.tensor(X_test_step, dtype=torch.float32, device=reg.device).unsqueeze(0)
            X_test_tensor = pad_x(X_test_tensor, reg.max_features)
            y_context_tensor = torch.tensor(y_train_step, dtype=torch.float32, device=reg.device).unsqueeze(0)

            preds = reg.model(
                x_src=torch.cat([X_train_tensor, X_test_tensor], dim=1),
                y_src=y_context_tensor.unsqueeze(-1),
                task="reg",
                text_enhanced_attn_weight=attn_weight_external,
            )
            preds = preds.squeeze(-1).reshape(-1)

            loss = torch.nn.functional.mse_loss(preds, y_target)
            if GATE_REG_STRENGTH and GATE_REG_STRENGTH > 0:
                gate_prob = torch.sigmoid(gate)
                gate_reg = (gate_prob * (1 - gate_prob)).mean()
                loss = loss + GATE_REG_STRENGTH * gate_reg

            loss.backward()
            optimizer.step()

            # Optional stability clamp in logit space (NOT in [0,1] space).
            if GATE_LOGIT_CLAMP is not None:
                with torch.no_grad():
                    gate.clamp_(-GATE_LOGIT_CLAMP, GATE_LOGIT_CLAMP)

            if (step_idx + 1) % STEP_LOG_EVERY == 0 or (step_idx == num_steps - 1):
                preds_np = preds.detach().cpu().numpy()
                mse = mean_squared_error(y_target.detach().cpu().numpy(), preds_np)
                rmse = float(np.sqrt(mse))
                mae = mean_absolute_error(y_target.detach().cpu().numpy(), preds_np)
                print(
                    f"Epoch {epoch:02d} Step {step_idx+1:03d}/{num_steps} "
                    f"(rows {start}–{end-1}) | MAE: {mae:.4f} | RMSE: {rmse:.4f} | {gate_stats(gate)}"
                )

    # (4) Freeze again to avoid surprises if you re-use `reg` later.
    for p in reg.model.parameters():
        p.requires_grad_(False)


def main() -> None:
    # Step 1: load the dataset
    X, y, text = load_climate_dataset(
        path=DATA_PATH,
        max_rows=MAX_ROWS,
        embedding_lags=TEXT_EMBEDDING_LAGS,
    )

    # Step 2: split chronologically
    (
        X_context,
        y_context,
        text_context,
        X_tune,
        y_tune,
        text_tune,
        X_eval,
        y_eval,
        text_eval,
    ) = time_split(X, y, text, context_ratio=CONTEXT_RATIO, tune_ratio=TUNE_RATIO)
    print(f"Split sizes: context={len(y_context)} tune={len(y_tune)} eval={len(y_eval)}")

    # Step 3: initialize regressor + store context set
    reg = load_tabdpt_regressor(device=DEVICE, model_weight_path=MODEL_WEIGHT_PATH, text_enhanced=True)
    reg.fit(X_context, y_context, text_context)

    reduction_mode, reduction_payload = prepare_feature_reduction(reg, seed=FEATURE_SUBSAMPLE_SEED)
    X_context_proc = preprocess_features(
        reg,
        X_context,
        reduction_mode=reduction_mode,
        reduction_payload=reduction_payload,
    )
    X_tune_proc = preprocess_features(
        reg,
        X_tune,
        reduction_mode=reduction_mode,
        reduction_payload=reduction_payload,
    )

    # Step 4: baseline eval
    print("\n== Baseline (before tuning) ==")
    baseline_no_text = evaluate_no_text_attn(reg, X=X_eval, y=y_eval, label="Eval (no text attn)")
    evaluate(reg, X=X_eval, y=y_eval, text=text_eval, label="Eval (with text attn)")
    if DEBUG_TEXT_EFFECT:
        compare_text_effect(
            reg,
            X=X_eval,
            text=text_eval,
            label="Eval text effect",
            max_rows=DEBUG_TEXT_EFFECT_ROWS,
        )
    print("Initial gate:", gate_stats(get_last_layer_gate_param(reg)))

    # Step 5: fine-tune gate on tune split
    print("\n== Fine-tuning alpha gate ==")
    fine_tune_external_gate(
        reg,
        X_context_proc=X_context_proc,
        y_context=y_context,
        text_context=text_context,
        X_tune_proc=X_tune_proc,
        y_tune=y_tune,
        text_tune=text_tune,
    )

    # Step 6: eval after tuning
    print("\n== After tuning ==")
    if FREEZE_NO_TEXT_BASELINE:
        _format_metrics("Eval (no text attn)", *baseline_no_text)
    else:
        evaluate_no_text_attn(reg, X=X_eval, y=y_eval, label="Eval (no text attn)")
    evaluate(reg, X=X_eval, y=y_eval, text=text_eval, label="Eval (with text attn)")
    if DEBUG_TEXT_EFFECT:
        compare_text_effect(
            reg,
            X=X_eval,
            text=text_eval,
            label="Eval text effect",
            max_rows=DEBUG_TEXT_EFFECT_ROWS,
        )
    print("Tuned gate:", gate_stats(get_last_layer_gate_param(reg)))


if __name__ == "__main__":
    main()


Split sizes: context=249 tune=748 eval=251

== Baseline (before tuning) ==
Eval (no text attn) | MAE: 19773.0527 | RMSE: 25539.1439
Eval (with text attn) | MAE: 19539.4648 | RMSE: 25322.7562
Eval text effect | Δpred(mean/max)=245.705780/316.302734
Initial gate: logits(mean/min/max)=0.0000/0.0000/0.0000 | sigmoid(mean/min/max)=0.5000/0.5000/0.5000 | sigmoid(sample)=[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

== Fine-tuning alpha gate ==
Tuning last-layer alpha gate: logits(mean/min/max)=0.0000/0.0000/0.0000 | sigmoid(mean/min/max)=0.5000/0.5000/0.5000 | sigmoid(sample)=[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
Epoch 01 Step 010/187 (rows 36–39) | MAE: 79.4301 | RMSE: 121.8134 | logits(mean/min/max)=-0.0366/-0.9291/0.8035 | sigmoid(mean/min/max)=0.4923/0.2831/0.6907 | sigmoid(sample)=[0.2890999913215637, 0.675000011920929, 0.3043000102043152, 0.2831000089645386, 0.6906999945640564, 0.6728000044822693, 0.6875, 0.3361000120639801]
Epoch 01 Step 020/187 (rows 76–79) | MAE: 103.5432 | RMSE: 133